In [ ]:
%run keys_99_2_getaccesskeys.ipynb

In [ ]:
import boto3
from botocore.exceptions import ClientError


class SmsSender:
    _client = None  # shared SNS client for the whole class

    @staticmethod
    def init_client(access_key=None, secret_key=None, region="us-east-1"):
        """Create the SNS client once. If keys are omitted, boto3 uses
        environment variables or your `aws configure` credentials."""
        SmsSender._client = boto3.client(
            "sns",
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            region_name=region
        )

    @staticmethod
    def create_sms_topic(topic_name):
        """Create the topic (or return the existing one) and return its ARN."""
        response = SmsSender._client.create_topic(Name=topic_name)
        return response["TopicArn"]

    @staticmethod
    def add_phone_numbers_to_topic(topic_arn, phone_numbers):
        """Subscribe a list of phone numbers (E.164, e.g. +15551234567) to a topic.
        Returns a dict: {phone_number: subscription_arn or None if it failed}."""
        results = {}
        for number in phone_numbers:
            try:
                response = SmsSender._client.subscribe(
                    TopicArn=topic_arn,
                    Protocol="sms",
                    Endpoint=number
                )
                results[number] = response["SubscriptionArn"]
                print(f"Subscribed {number}")
            except ClientError as e:
                results[number] = None
                print(f"Failed to subscribe {number}: {e.response['Error']['Message']}")
        return results

    @staticmethod
    def publish_to_topic(topic_arn, message, sms_type="Transactional"):
        """Publish a message to a topic. Every subscribed phone number receives it as an SMS.
        Returns the MessageId, or None if it failed."""
        try:
            response = SmsSender._client.publish(
                TopicArn=topic_arn,
                Message=message,
                MessageAttributes={
                    "AWS.SNS.SMS.SMSType": {
                        "DataType": "String",
                        "StringValue": sms_type  # "Transactional" or "Promotional"
                    }
                }
            )
            print("Message published. Message ID:", response["MessageId"])
            return response["MessageId"]
        except ClientError as e:
            print("Failed to publish:", e.response["Error"]["Message"])
            return None




In [ ]:
# Connecting
SmsSender.init_client(
    access_key=keyname,
    secret_key=secretkey,
    region="us-east-1"
)
print("SmsSender.init_client connection ->successful")
print(SmsSender._client)

In [ ]:
# 1 Create a Topic.
topic_arn = SmsSender.create_sms_topic("mysmstopic1")

In [ ]:
# 2. Add Phone NUmbers to a topic
phone_numbers = ["+xx", "yy"]
SmsSender.add_phone_numbers_to_topic(topic_arn, phone_numbers)

In [ ]:
# 3. Publish a message to everyone on the topic
SmsSender.publish_to_topic(topic_arn, "Hello! This is a test message from SMS.")

In [ ]:
# for debugging...

import boto3

def show(label, client):
    creds = client._request_signer._credentials
    key = creds.access_key
    print(f"{label}: {key[:4]}...{key[-4:]} (length {len(key)}), token set: {bool(creds.token)}")

# Your existing S3 client (however you create it today)
s3 = boto3.client("s3")   # replace with your actual S3 client creation
show("S3 client", s3)

# The SNS client from your class
show("SNS client", SmsSender._client)